# Lab 2 Phase 7B: Modal Residual-Target L1 SR

Modal-only notebook for calibrated super-resolution architecture ablations.

- Data mount: `/mnt/data` with paired Lab 2 data and COCO/ImageNet archives under `Data/`.
- Output mount: `/mnt/runs`, defaulting to `/mnt/runs/phase7b_npu_model_ablation`.
- Models learn `delta = HR - LR` using pure residual-target L1 loss.
- Inference and ONNX export output reconstructed SR: `LR + delta_pred`.


In [2]:
%uv pip install -q onnx onnxruntime nbformat pillow torchvision

Note: you may need to restart the kernel to use updated packages.


In [3]:
from __future__ import annotations

from collections import defaultdict
from contextlib import nullcontext
from pathlib import Path
from torch.utils.data import WeightedRandomSampler
from torchvision import transforms
from typing import Any
import copy
import hashlib
import io
import json
import math
import os
import random
import time
import warnings
import zipfile

warnings.filterwarnings("ignore", message=".*legacy TorchScript-based ONNX.*")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageFilter, ImageOps
from torch.optim import AdamW
from torch.utils.data import ConcatDataset, DataLoader, Dataset, Subset

try:
    import onnx
except ImportError:
    onnx = None

try:
    import onnxruntime as ort
except ImportError:
    ort = None

TO_TENSOR = transforms.ToTensor()
TO_PIL = transforms.ToPILImage()
BICUBIC = Image.Resampling.BICUBIC if hasattr(Image, "Resampling") else Image.BICUBIC
BILINEAR = Image.Resampling.BILINEAR if hasattr(Image, "Resampling") else Image.BILINEAR
LANCZOS = Image.Resampling.LANCZOS if hasattr(Image, "Resampling") else Image.LANCZOS
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

MOBILINT_NPU_SUPPORTED_ONNX = {
    "AMax", "BatchNormalization", "Celu", "Conv", "DepthToSpace", "Einsum", "Elu", "Embedding",
    "FloorDiv", "FloorMod", "Gemm", "GlobalAveragePool", "HardSigmoid", "HardSwish", "Hardtanh",
    "Identity", "InstanceNormalization", "LeakyRelu", "MaskedFill", "Mish", "Neg", "Pad", "PRelu",
    "MaxPool", "AveragePool", "Pow", "QuickGelu", "ReduceL2", "Repeat", "Roll", "Split", "SplitV2",
    "Sub", "Swish", "TopK", "ConvTranspose", "Unflatten", "Upsample",
}
MOBILINT_CPU_FALLBACK_ONNX = {
    "Add", "Cast", "Clip", "Concat", "Div", "Erf", "Exp", "Expand", "Flatten", "Gather", "GatherND",
    "Gelu", "Greater", "GroupNormalization", "MatMul", "Max", "Mean", "Min", "Mul", "ReduceMax",
    "ReduceMean", "ReduceMin", "ReduceProd", "ReduceSum", "Relu", "Reshape", "Resize", "ScatterND",
    "Sigmoid", "Slice", "Softmax", "Softplus", "Squeeze", "Tanh", "Tile", "Transpose", "Unsqueeze", "Where",
}
FORBIDDEN_MODULE_TYPES = (nn.ReLU, nn.Sigmoid, nn.Softmax, nn.LayerNorm, nn.GroupNorm)


In [4]:
# Configuration
OUTPUT_SUBDIR = "phase7b_npu_model_ablation"
MODEL_ID = os.environ.get("LAB2_PHASE7B_MODEL_ID", "mixed_kernel_residual")
BATCH_SIZE = int(os.environ.get("LAB2_BATCH_SIZE", "24"))
NUM_WORKERS = int(os.environ.get("LAB2_NUM_WORKERS", "8"))
PREFETCH_FACTOR = int(os.environ.get("LAB2_PREFETCH_FACTOR", "4"))
SEED = int(os.environ.get("LAB2_SEED", "255"))
TRAIN_PATCH_SIZE = int(os.environ.get("LAB2_TRAIN_PATCH_SIZE", "224"))
EVAL_SIZE = int(os.environ.get("LAB2_EVAL_SIZE", "256"))
CHANNELS_LAST = os.environ.get("LAB2_CHANNELS_LAST", "1").strip().lower() not in {"0", "false", "no"}
USE_AMP = os.environ.get("LAB2_USE_AMP", "1").strip().lower() not in {"0", "false", "no"}
TORCH_COMPILE = os.environ.get("LAB2_TORCH_COMPILE", "0").strip().lower() not in {"0", "false", "no"}
RESUME_TRAINING = os.environ.get("LAB2_RESUME_TRAINING", "1").strip().lower() not in {"0", "false", "no"}
RUN_FULL_TRAINING = os.environ.get("LAB2_RUN_FULL_TRAINING", "1").strip().lower() not in {"0", "false", "no"}
RUN_ONNX_EXPORT = os.environ.get("LAB2_RUN_ONNX_EXPORT", "1").strip().lower() not in {"0", "false", "no"}
FAIL_ON_EASY_SYNTHETIC = os.environ.get("LAB2_FAIL_ON_EASY_SYNTHETIC", "0").strip().lower() not in {"0", "false", "no"}
SAVE_EPOCH_CHECKPOINTS = os.environ.get("LAB2_SAVE_EPOCH_CHECKPOINTS", "0").strip().lower() not in {"0", "false", "no"}
SYNTHETIC_VAL_CACHE = os.environ.get("LAB2_SYNTHETIC_VAL_CACHE", "1").strip().lower() not in {"0", "false", "no"}
SYNTHETIC_EVAL_INTERVAL = max(1, int(os.environ.get("LAB2_SYNTHETIC_EVAL_INTERVAL", "4")))
SYNTHETIC_TOO_EASY_PSNR = float(os.environ.get("LAB2_SYNTHETIC_TOO_EASY_PSNR", "25.0"))
MAX_STEPS_PER_EPOCH = int(os.environ.get("LAB2_MAX_STEPS_PER_EPOCH", "0"))
MAX_EVAL_BATCHES = int(os.environ.get("LAB2_MAX_EVAL_BATCHES", "0"))
MIN_METRIC_IMPROVEMENT = float(os.environ.get("LAB2_MIN_METRIC_IMPROVEMENT", "0.01"))

STAGE_EPOCH_OVERRIDES = os.environ.get("LAB2_STAGE_EPOCHS")
if STAGE_EPOCH_OVERRIDES:
    parts = [int(x.strip()) for x in STAGE_EPOCH_OVERRIDES.split(",") if x.strip()]
    if len(parts) != 3:
        raise ValueError("LAB2_STAGE_EPOCHS must be three comma-separated integers, e.g. 10,20,12")
    STAGE_EPOCHS = {"synthetic_warmup": parts[0], "mixed_finetune": parts[1], "paired_polish": parts[2]}
else:
    STAGE_EPOCHS = {"synthetic_warmup": 10, "mixed_finetune": 20, "paired_polish": 12}

DATA_CFG = {
    "train_patch_size": TRAIN_PATCH_SIZE,
    "eval_size": EVAL_SIZE,
    "random_scale_pad": 64,
    "imagenet_train_limit": int(os.environ.get("LAB2_IMAGENET_TRAIN_LIMIT", "6000")),
    "imagenet_val_limit": int(os.environ.get("LAB2_IMAGENET_VAL_LIMIT", "300")),
    "coco_train_limit": int(os.environ.get("LAB2_COCO_TRAIN_LIMIT", "12000")),
    "coco_val_limit": int(os.environ.get("LAB2_COCO_VAL_LIMIT", "500")),
    "train_eval_subset_size": int(os.environ.get("LAB2_TRAIN_EVAL_SUBSET_SIZE", "256")),
    "target_psnr_low": float(os.environ.get("LAB2_TARGET_PSNR_LOW", "17.5")),
    "target_psnr_high": float(os.environ.get("LAB2_TARGET_PSNR_HIGH", "24.2")),
    "target_psnr_center": float(os.environ.get("LAB2_TARGET_PSNR_CENTER", "21.2")),
    "stage2_target_psnr_low": float(os.environ.get("LAB2_STAGE2_TARGET_PSNR_LOW", "18.0")),
    "stage2_target_psnr_high": float(os.environ.get("LAB2_STAGE2_TARGET_PSNR_HIGH", "23.5")),
    "target_grad_ratio_low": float(os.environ.get("LAB2_TARGET_GRAD_RATIO_LOW", "0.35")),
    "target_grad_ratio_high": float(os.environ.get("LAB2_TARGET_GRAD_RATIO_HIGH", "0.85")),
    "target_lap_ratio_low": float(os.environ.get("LAB2_TARGET_LAP_RATIO_LOW", "0.25")),
    "target_lap_ratio_high": float(os.environ.get("LAB2_TARGET_LAP_RATIO_HIGH", "0.75")),
    "max_degrade_attempts": int(os.environ.get("LAB2_MAX_DEGRADE_ATTEMPTS", "12")),
    "patch_mining_candidates": int(os.environ.get("LAB2_PATCH_MINING_CANDIDATES", "8")),
    "synthetic_patch_mining_candidates": int(os.environ.get("LAB2_SYNTHETIC_PATCH_MINING_CANDIDATES", "2")),
    "hard_image_threshold": float(os.environ.get("LAB2_HARD_IMAGE_THRESHOLD", "24.8")),
    "hard_patch_weight": float(os.environ.get("LAB2_HARD_PATCH_WEIGHT", "2.0")),
    "paired_fraction_mixed": float(os.environ.get("LAB2_PAIRED_FRACTION_MIXED", "0.65")),
    "synthetic_val_cache": SYNTHETIC_VAL_CACHE,
}

TRAIN_CFG_BY_STAGE = {
    "synthetic_warmup": {
        "stage_name": "synthetic_warmup", "epochs": STAGE_EPOCHS["synthetic_warmup"], "lr": 3e-4,
        "weight_decay": 1e-4, "warmup_epochs": 2, "min_lr_ratio": 0.08, "grad_clip_norm": 1.0,
        "ema_decay": 0.999, "selection_metric": "paired_val_psnr", "checkpoint_interval": 4,
        "train_eval_interval": 2, "early_stop_patience": 8,
    },
    "mixed_finetune": {
        "stage_name": "mixed_finetune", "epochs": STAGE_EPOCHS["mixed_finetune"], "lr": 1.5e-4,
        "weight_decay": 5e-5, "warmup_epochs": 1, "min_lr_ratio": 0.05, "grad_clip_norm": 1.0,
        "ema_decay": 0.999, "selection_metric": "paired_val_psnr", "checkpoint_interval": 4,
        "train_eval_interval": 1, "early_stop_patience": 10,
    },
    "paired_polish": {
        "stage_name": "paired_polish", "epochs": STAGE_EPOCHS["paired_polish"], "lr": 5e-5,
        "weight_decay": 0.0, "warmup_epochs": 1, "min_lr_ratio": 0.10, "grad_clip_norm": 0.75,
        "ema_decay": 0.9995, "selection_metric": "paired_val_psnr", "checkpoint_interval": 2,
        "train_eval_interval": 1, "early_stop_patience": 8,
    },
}

CALIBRATION_CFG = {"num_samples": int(os.environ.get("LAB2_CALIBRATION_SAMPLES", "128")), "seed": SEED, "output_subdir": "calibration"}

print(json.dumps({
    "model_id": MODEL_ID,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "prefetch_factor": PREFETCH_FACTOR,
    "stage_epochs": STAGE_EPOCHS,
    "torch_compile": TORCH_COMPILE,
    "run_full_training": RUN_FULL_TRAINING,
    "run_onnx_export": RUN_ONNX_EXPORT,
    "fail_on_easy_synthetic": FAIL_ON_EASY_SYNTHETIC,
}, indent=2))


{
  "model_id": "mixed_kernel_residual",
  "batch_size": 24,
  "num_workers": 8,
  "prefetch_factor": 4,
  "stage_epochs": {
    "synthetic_warmup": 10,
    "mixed_finetune": 20,
    "paired_polish": 12
  },
  "torch_compile": false,
  "run_full_training": true,
  "run_onnx_export": true,
  "fail_on_easy_synthetic": false
}


In [5]:
# Runtime and workspace helpers
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def configure_runtime() -> torch.device:
    if torch.cuda.is_available():
        device = torch.device("cuda")
        torch.backends.cudnn.benchmark = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        try:
            torch.set_float32_matmul_precision("high")
        except AttributeError:
            pass
    elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    return device


def choose_amp_policy(device: torch.device) -> dict[str, Any]:
    if device.type != "cuda" or not USE_AMP:
        return {"enabled": False, "dtype": None, "use_scaler": False, "label": "fp32"}
    bf16_ok = False
    if hasattr(torch.cuda, "is_bf16_supported"):
        try:
            bf16_ok = torch.cuda.is_bf16_supported()
        except Exception:
            bf16_ok = False
    if bf16_ok:
        return {"enabled": True, "dtype": torch.bfloat16, "use_scaler": False, "label": "bf16"}
    return {"enabled": True, "dtype": torch.float16, "use_scaler": True, "label": "fp16"}


def make_grad_scaler(amp_policy: dict[str, Any]):
    if not amp_policy["enabled"] or not amp_policy["use_scaler"]:
        return None
    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        return torch.amp.GradScaler("cuda")
    return torch.cuda.amp.GradScaler()


def autocast_context(device: torch.device, amp_policy: dict[str, Any]):
    if device.type != "cuda" or not amp_policy["enabled"]:
        return nullcontext()
    if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):
        return torch.amp.autocast("cuda", dtype=amp_policy["dtype"])
    return torch.cuda.amp.autocast(dtype=amp_policy["dtype"])


def first_existing(*paths: Path) -> Path:
    for path in paths:
        if path.exists():
            return path
    return paths[0]


def zip_image_member_count(zip_path: Path) -> int:
    with zipfile.ZipFile(zip_path) as zf:
        return sum(1 for info in zf.infolist() if not info.is_dir() and Path(info.filename).suffix.lower() in IMAGE_SUFFIXES)


def extracted_image_count(extracted_dir: Path) -> int:
    if not extracted_dir.exists():
        return 0
    return sum(1 for path in extracted_dir.rglob("*") if path.suffix.lower() in IMAGE_SUFFIXES)


def zip_extract_marker(zip_path: Path) -> Path:
    return zip_path.parent / f".{zip_path.stem}.extract_complete"


def extract_zip_if_needed(zip_path: Path, extracted_dir: Path) -> Path:
    if not zip_path.exists():
        return extracted_dir
    marker = zip_extract_marker(zip_path)
    if marker.exists() and extracted_dir.exists():
        return extracted_dir
    needs_extract = not extracted_dir.exists()
    if not needs_extract:
        expected = zip_image_member_count(zip_path)
        actual = extracted_image_count(extracted_dir)
        needs_extract = expected > 0 and actual < expected
    if needs_extract:
        print(f"Extracting {zip_path.name} into {zip_path.parent} ...")
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(zip_path.parent)
    marker.write_text("ok\n")
    return extracted_dir


def ensure_unzipped(zip_path: Path, extracted_dir: Path) -> Path:
    return extract_zip_if_needed(zip_path, extracted_dir)


def resolve_workspace(output_subdir: str) -> dict[str, Path | bool]:
    data_override = os.environ.get("LAB2_DATA_ROOT")
    output_override = os.environ.get("LAB2_OUTPUT_DIR")
    workspace_root = Path.cwd().resolve()

    def has_paired_layout(root: Path) -> bool:
        return (
            (root / "HR_train").exists() and (root / "LR_train").exists()
        ) or (
            (root / "train" / "HR").exists() and (root / "train" / "LR").exists()
        )

    def candidate_variants(base: Path) -> list[Path]:
        variants = [base, base / "Data", base / "data"]
        if base.exists():
            for child in sorted(p for p in base.iterdir() if p.is_dir()):
                variants.append(child)
                # Some Modal volumes get mounted with one extra nesting layer.
                if child.name.lower() in {"data", "dataset", "datasets", "lab2", "lab_2", "volumes", "mounts"}:
                    variants.extend([child / "Data", child / "data"])
        return variants

    def dedupe_paths(paths: list[Path]) -> list[Path]:
        out: list[Path] = []
        seen: set[str] = set()
        for p in paths:
            key = str(p)
            if key not in seen:
                out.append(p)
                seen.add(key)
        return out

    search_bases = [
        Path("/mnt/data"),
        Path("/mnt/lab2-phase7-data"),
        Path("/mnt/lab-2-phase-7-data"),
        Path("/data"),
        Path("/root/data"),
        Path("/root"),
        Path("/vol/data"),
        Path("/workspace"),
        workspace_root,
    ]
    candidate_roots: list[Path] = []
    if data_override:
        override = Path(data_override).expanduser().resolve()
        candidate_roots.extend(candidate_variants(override))
    for base in search_bases:
        candidate_roots.extend(candidate_variants(base))

    deduped = dedupe_paths(candidate_roots)

    # First pass: direct candidates.
    for candidate in deduped:
        if has_paired_layout(candidate):
            data_root = candidate
            break
    else:
        # Second pass: shallow descendants under existing candidates.
        found = None
        for base in [p for p in deduped if p.exists()]:
            try:
                for child in sorted(p for p in base.iterdir() if p.is_dir()):
                    if has_paired_layout(child):
                        found = child
                        break
            except (PermissionError, OSError):
                continue
            if found is not None:
                break
        if found is not None:
            data_root = found
        else:
            existing = [p for p in deduped if p.exists()]
            data_root = existing[0] if existing else Path("/mnt/data")

    default_output_root = Path("/mnt/runs") if Path("/mnt/runs").exists() else workspace_root / "runs"
    output_dir = Path(output_override).expanduser().resolve() if output_override else (default_output_root / output_subdir)
    output_dir.mkdir(parents=True, exist_ok=True)

    return {
        "repo_root": workspace_root,
        "data_root": data_root,
        "output_dir": output_dir,
        "workspace_root": workspace_root,
        "in_modal_notebook": True,
    }


def slugify_name(text: str) -> str:
    cleaned = "".join(ch if ch.isalnum() else "_" for ch in str(text))
    return "_".join(part for part in cleaned.split("_") if part)[:96] or "sample"


set_seed(SEED)
device = configure_runtime()
amp_policy = choose_amp_policy(device)
channels_last = bool(CHANNELS_LAST and device.type == "cuda")
workspace = resolve_workspace(OUTPUT_SUBDIR)
OUTPUT_DIR = Path(workspace["output_dir"])
DATA_ROOT = Path(workspace["data_root"])
print(f"Device: {device} | AMP: {amp_policy['label']} | channels_last={channels_last}")
print(f"Data root: {DATA_ROOT}")
print(f"Output dir: {OUTPUT_DIR}")


Device: cuda | AMP: bf16 | channels_last=True
Data root: /mnt/lab2-phase7-data/Data
Output dir: /root/runs/phase7b_npu_model_ablation


In [6]:
# Data discovery and degradation metrics
def collect_paired_by_subfolder(lr_root: Path, hr_root: Path) -> list[tuple[Path, Path, str]]:
    pairs: list[tuple[Path, Path, str]] = []
    if not lr_root.exists() or not hr_root.exists():
        return pairs
    for hr_dir in sorted(p for p in hr_root.iterdir() if p.is_dir()):
        suffix = hr_dir.name.replace("HR_train", "")
        lr_dir = lr_root / f"LR_train{suffix}"
        if not lr_dir.exists():
            continue
        hr_imgs = {p.stem: p for p in sorted(hr_dir.glob("*.png"))}
        lr_imgs = {p.stem: p for p in sorted(lr_dir.glob("*.png"))}
        pairs.extend((lr_imgs[s], hr_imgs[s], f"{hr_dir.name}/{s}") for s in sorted(set(hr_imgs) & set(lr_imgs)))
    return pairs


def collect_paired_flat(lr_dir: Path, hr_dir: Path) -> list[tuple[Path, Path, str]]:
    if not lr_dir.exists() or not hr_dir.exists():
        return []
    hr_imgs = {p.stem: p for p in sorted(hr_dir.glob("*.png"))}
    lr_imgs = {p.stem: p for p in sorted(lr_dir.glob("*.png"))}
    return [(lr_imgs[s], hr_imgs[s], s) for s in sorted(set(hr_imgs) & set(lr_imgs))]


def collect_train_pairs(data_root: Path) -> list[tuple[Path, Path, str]]:
    structured = collect_paired_by_subfolder(data_root / "LR_train", data_root / "HR_train")
    if structured:
        return structured
    return collect_paired_flat(data_root / "train" / "LR", data_root / "train" / "HR")


def collect_val_pairs(data_root: Path) -> list[tuple[Path, Path, str]]:
    for lr_dir, hr_dir in [
        (data_root / "LR_val", data_root / "HR_val"),
        (data_root / "val" / "LR_val", data_root / "val" / "HR_val"),
        (data_root / "val" / "LR", data_root / "val" / "HR"),
    ]:
        pairs = collect_paired_flat(lr_dir, hr_dir)
        if pairs:
            return pairs
    return []


def pil_rgb(path: Path) -> Image.Image:
    return Image.open(path).convert("RGB")


def tensor_psnr(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    mse = torch.mean((a.float().clamp(0, 1) - b.float().clamp(0, 1)) ** 2, dim=(-3, -2, -1)).clamp_min(1e-12)
    return -10.0 * torch.log10(mse)


def pil_to_np(img: Image.Image) -> np.ndarray:
    return np.asarray(img.convert("RGB"), dtype=np.float32) / 255.0


def np_psnr(a: np.ndarray, b: np.ndarray) -> float:
    mse = float(np.mean((a - b) ** 2))
    return -10.0 * math.log10(max(mse, 1e-12))


def gray_np(a: np.ndarray) -> np.ndarray:
    return 0.299 * a[:, :, 0] + 0.587 * a[:, :, 1] + 0.114 * a[:, :, 2]


def grad_energy_np(a: np.ndarray) -> float:
    g = gray_np(a)
    return float(0.5 * (np.abs(g[:, 1:] - g[:, :-1]).mean() + np.abs(g[1:, :] - g[:-1, :]).mean()))


def lap_energy_np(a: np.ndarray) -> float:
    g = gray_np(a)
    lap = -4 * g[1:-1, 1:-1] + g[:-2, 1:-1] + g[2:, 1:-1] + g[1:-1, :-2] + g[1:-1, 2:]
    return float(np.abs(lap).mean())


def pair_metric_row(lr_img: Image.Image, hr_img: Image.Image, name: str, source: str) -> dict[str, Any]:
    lr = pil_to_np(lr_img)
    hr = pil_to_np(hr_img)
    residual = hr - lr
    lr_grad, hr_grad = grad_energy_np(lr), grad_energy_np(hr)
    lr_lap, hr_lap = lap_energy_np(lr), lap_energy_np(hr)
    return {
        "name": name,
        "source": source,
        "baseline_psnr": np_psnr(lr, hr),
        "mean_abs_residual": float(np.mean(np.abs(residual))),
        "signed_residual_mean_rgb": [float(x) for x in np.mean(residual, axis=(0, 1))],
        "lr_grad_energy": lr_grad,
        "hr_grad_energy": hr_grad,
        "grad_ratio": lr_grad / max(hr_grad, 1e-12),
        "lr_lap_energy": lr_lap,
        "hr_lap_energy": hr_lap,
        "lap_ratio": lr_lap / max(hr_lap, 1e-12),
    }


def summarize_numeric(rows: list[dict[str, Any]], keys: list[str]) -> dict[str, Any]:
    summary: dict[str, Any] = {"count": len(rows)}
    for key in keys:
        vals = sorted(float(row[key]) for row in rows if key in row)
        if not vals:
            continue
        def quantile(q: float) -> float:
            return vals[min(len(vals) - 1, max(0, round(q * (len(vals) - 1))))]
        summary[key] = {
            "mean": float(np.mean(vals)),
            "std": float(np.std(vals)),
            "p10": quantile(0.10),
            "p25": quantile(0.25),
            "p50": quantile(0.50),
            "p75": quantile(0.75),
            "p90": quantile(0.90),
            "min": vals[0],
            "max": vals[-1],
        }
    return summary


def build_degradation_profile(train_pairs: list[tuple[Path, Path, str]], val_pairs: list[tuple[Path, Path, str]], output_dir: Path) -> dict[str, Any]:
    rows_by_split: dict[str, list[dict[str, Any]]] = {}
    for split, pairs in {"paired_train": train_pairs, "paired_val": val_pairs}.items():
        rows = []
        for lr_path, hr_path, name in pairs:
            rows.append(pair_metric_row(pil_rgb(lr_path), pil_rgb(hr_path), name=name, source=split))
        rows_by_split[split] = rows

    keys = ["baseline_psnr", "mean_abs_residual", "grad_ratio", "lap_ratio", "lr_grad_energy", "hr_grad_energy", "lr_lap_energy", "hr_lap_energy"]
    profile = {
        "created_at_unix": time.time(),
        "config": {
            "target_psnr_low": DATA_CFG["target_psnr_low"],
            "target_psnr_high": DATA_CFG["target_psnr_high"],
            "target_psnr_center": DATA_CFG["target_psnr_center"],
        },
        "summaries": {split: summarize_numeric(rows, keys) for split, rows in rows_by_split.items()},
        "paired_train_rows": rows_by_split["paired_train"],
        "paired_val_rows": rows_by_split["paired_val"],
    }
    output_path = output_dir / "degradation_profile.json"
    output_path.write_text(json.dumps(profile, indent=2))
    print(f"Wrote degradation profile: {output_path}")
    for split in ["paired_train", "paired_val"]:
        s = profile["summaries"][split]
        print(f"{split}: n={s['count']} | baseline_psnr mean={s['baseline_psnr']['mean']:.3f} p10={s['baseline_psnr']['p10']:.3f} p50={s['baseline_psnr']['p50']:.3f} p90={s['baseline_psnr']['p90']:.3f}")
        print(f"  residual mean={s['mean_abs_residual']['mean']:.5f} | grad_ratio mean={s['grad_ratio']['mean']:.3f} | lap_ratio mean={s['lap_ratio']['mean']:.3f}")
    return profile


def apply_profile_targets(profile: dict[str, Any], cfg: dict[str, Any]) -> None:
    if os.environ.get("LAB2_DISABLE_PROFILE_TARGETS", "0").strip().lower() in {"1", "true", "yes"}:
        return
    val_summary = profile["summaries"].get("paired_val", {})
    psnr = val_summary.get("baseline_psnr", {})
    residual = val_summary.get("mean_abs_residual", {})
    if psnr:
        cfg["target_psnr_center"] = float(psnr.get("p50", psnr.get("mean", cfg["target_psnr_center"])))
        cfg["target_psnr_low"] = float(max(16.5, psnr.get("p10", cfg["target_psnr_low"])))
        cfg["target_psnr_high"] = float(min(24.2, psnr.get("p90", cfg["target_psnr_high"])))
    if residual:
        cfg["target_mean_abs_residual"] = float(residual.get("mean", 0.0))
    rgb_rows = profile.get("paired_val_rows", [])
    if rgb_rows:
        cfg["target_rgb_residual_mean"] = [float(x) for x in np.mean([row["signed_residual_mean_rgb"] for row in rgb_rows], axis=0)]
    cfg["stage2_target_psnr_low"] = float(max(18.0, min(cfg["stage2_target_psnr_low"], cfg["target_psnr_center"] - 2.0)))
    cfg["stage2_target_psnr_high"] = float(min(23.5, cfg["target_psnr_high"]))
    print("Profile-derived synthetic targets:", json.dumps({
        "target_psnr_low": cfg["target_psnr_low"],
        "target_psnr_center": cfg["target_psnr_center"],
        "target_psnr_high": cfg["target_psnr_high"],
        "target_mean_abs_residual": cfg.get("target_mean_abs_residual"),
        "target_rgb_residual_mean": cfg.get("target_rgb_residual_mean"),
    }, indent=2))


train_pairs = collect_train_pairs(DATA_ROOT)
val_pairs = collect_val_pairs(DATA_ROOT)
if not train_pairs:
    hints = [
        DATA_ROOT,
        DATA_ROOT / "Data",
        DATA_ROOT / "data",
        Path("/mnt/data"),
        Path("/data"),
        Path("/root/data"),
        Path("/workspace"),
    ]
    existing = [str(p) for p in hints if p.exists()]
    nearby_dirs = []
    for h in hints:
        if not h.exists() or not h.is_dir():
            continue
        try:
            nearby_dirs.extend(str(p) for p in sorted(c for c in h.iterdir() if c.is_dir())[:12])
        except (PermissionError, OSError):
            continue
    raise FileNotFoundError(
        "No paired training images found. "
        f"Resolved DATA_ROOT={DATA_ROOT}. Existing nearby paths: {existing}. "
        f"Sample nearby directories: {nearby_dirs[:24]}. "
        "Expected either HR_train/LR_train or train/HR + train/LR under DATA_ROOT."
    )
if not val_pairs:
    raise FileNotFoundError(
        "No paired validation images found. "
        f"Resolved DATA_ROOT={DATA_ROOT}. "
        "Expected either LR_val/HR_val or val/LR_val + val/HR_val (or val/LR + val/HR)."
    )
print(f"Paired train pairs: {len(train_pairs)} | paired val pairs: {len(val_pairs)}")
degradation_profile = build_degradation_profile(train_pairs, val_pairs, OUTPUT_DIR)
apply_profile_targets(degradation_profile, DATA_CFG)


Paired train pairs: 3036 | paired val pairs: 100
Wrote degradation profile: /root/runs/phase7b_npu_model_ablation/degradation_profile.json
paired_train: n=3036 | baseline_psnr mean=26.757 p10=20.048 p50=26.120 p90=34.112
  residual mean=0.03912 | grad_ratio mean=0.631 | lap_ratio mean=0.457
paired_val: n=100 | baseline_psnr mean=21.336 p10=17.970 p50=21.565 p90=24.072
  residual mean=0.06237 | grad_ratio mean=0.598 | lap_ratio mean=0.449
Profile-derived synthetic targets: {
  "target_psnr_low": 17.969536692119192,
  "target_psnr_center": 21.565338311938834,
  "target_psnr_high": 24.071796771007662,
  "target_mean_abs_residual": 0.062371243312954905,
  "target_rgb_residual_mean": [
    0.0014350442632712656,
    0.001680861000513687,
    0.0015503103228547844
  ]
}


In [7]:
# Natural image manifests and calibrated degradation
def read_manifest_lines(path: Path) -> list[str]:
    if not path.exists():
        return []
    return [line.strip() for line in path.read_text().splitlines() if line.strip()]


def read_imagenet_manifest(path: Path) -> list[tuple[str, int]]:
    if not path.exists():
        return []
    rows = []
    for line in path.read_text().splitlines():
        parts = line.split()
        if len(parts) >= 2:
            rows.append((parts[0], int(parts[1])))
    return rows


def collect_imagenet_records(data_root: Path, split: str) -> list[dict[str, Any]]:
    course = data_root / "course_files_export"
    legacy = data_root / "ImageNet"
    if split == "train":
        list_path = first_existing(course / "imagenet_train20.txt", legacy / "imagenet_train20.txt")
        image_root = ensure_unzipped(course / "imagenet_train20.zip", first_existing(course / "imagenet_train20a", legacy / "imagenet_train20a"))
    else:
        list_path = first_existing(course / "imagenet_val20.txt", legacy / "imagenet_val20.txt")
        image_root = ensure_unzipped(course / "imagenet_val20.zip", first_existing(course / "imagenet_val20", legacy / "imagenet_val20"))
    records: list[dict[str, Any]] = []
    for filename, class_id in read_imagenet_manifest(list_path):
        synset = filename.split("_")[0]
        path = (image_root / synset / filename) if split == "train" else (image_root / filename)
        if path.exists():
            records.append({"path": path, "stem": path.stem, "source_name": f"imagenet_{split}", "class_id": class_id})
    return records


def coco_root(data_root: Path) -> Path:
    return data_root / "course_files_export" / "coco2017"


def ensure_coco_split(root: Path, split: str) -> tuple[Path, Path]:
    image_dir = root / f"{split}2017"
    zip_path = root / f"{split}2017.zip"
    manifest = root / f"coco_{split}2017.txt"
    if zip_path.exists():
        # Modal data volumes may store COCO as zips to avoid slow file-by-file volume clones.
        # The helper completes partial extractions, then writes a marker to skip future re-extracts.
        root.mkdir(parents=True, exist_ok=True)
        extract_zip_if_needed(zip_path, image_dir)
    if image_dir.exists():
        rels = [str(p.relative_to(root)) for p in sorted(image_dir.rglob("*")) if p.suffix.lower() in IMAGE_SUFFIXES]
        manifest.write_text("\n".join(rels) + ("\n" if rels else ""))
    return image_dir, manifest


def collect_coco_records(data_root: Path, split: str) -> list[dict[str, Any]]:
    root = coco_root(data_root)
    image_dir, manifest = ensure_coco_split(root, split)
    records = []
    for rel in read_manifest_lines(manifest):
        path = root / rel
        if path.exists():
            records.append({"path": path, "stem": path.stem, "source_name": f"coco_{split}"})
    if not records and image_dir.exists():
        records = [
            {"path": path, "stem": path.stem, "source_name": f"coco_{split}"}
            for path in sorted(image_dir.rglob("*"))
            if path.suffix.lower() in IMAGE_SUFFIXES
        ]
    return records


def take_subset(records: list[Any], limit: int | None, seed: int) -> list[Any]:
    if limit is None or limit >= len(records):
        return list(records)
    rng = random.Random(seed)
    items = list(records)
    rng.shuffle(items)
    return items[:limit]


def build_natural_records(data_root: Path, data_cfg: dict[str, Any], seed: int) -> dict[str, list[dict[str, Any]]]:
    imagenet_train = take_subset(collect_imagenet_records(data_root, "train"), data_cfg["imagenet_train_limit"], seed)
    imagenet_val = take_subset(collect_imagenet_records(data_root, "val"), data_cfg["imagenet_val_limit"], seed)
    coco_train = take_subset(collect_coco_records(data_root, "train"), data_cfg["coco_train_limit"], seed + 17)
    coco_val = take_subset(collect_coco_records(data_root, "val"), data_cfg["coco_val_limit"], seed + 17)
    train = coco_train + imagenet_train
    val = coco_val + imagenet_val
    if not train:
        raise FileNotFoundError("No COCO/ImageNet natural images found for calibrated synthetic training.")
    print(f"Natural records: train={len(train)} (COCO={len(coco_train)}, ImageNet={len(imagenet_train)}) | val={len(val)}")
    return {"train": train, "val": val, "coco_train": coco_train, "imagenet_train": imagenet_train, "coco_val": coco_val, "imagenet_val": imagenet_val}


def seeded_rng(key: str) -> random.Random:
    digest = hashlib.sha256(key.encode("utf-8")).hexdigest()
    return random.Random(int(digest[:16], 16))


def random_crop_single(img: Image.Image, size: int, rng: random.Random) -> Image.Image:
    if img.width < size or img.height < size:
        img = ImageOps.fit(img, (size, size), method=BICUBIC)
    if img.width == size and img.height == size:
        return img
    x0 = rng.randint(0, img.width - size)
    y0 = rng.randint(0, img.height - size)
    return img.crop((x0, y0, x0 + size, y0 + size))


def random_crop_pair(lr_img: Image.Image, hr_img: Image.Image, size: int, rng: random.Random) -> tuple[Image.Image, Image.Image]:
    if lr_img.width < size or lr_img.height < size:
        lr_img = ImageOps.fit(lr_img, (size, size), method=BICUBIC)
        hr_img = ImageOps.fit(hr_img, (size, size), method=BICUBIC)
    if lr_img.width == size and lr_img.height == size:
        return lr_img, hr_img
    x0 = rng.randint(0, lr_img.width - size)
    y0 = rng.randint(0, lr_img.height - size)
    box = (x0, y0, x0 + size, y0 + size)
    return lr_img.crop(box), hr_img.crop(box)


def hard_mine_crop_pair(lr_img: Image.Image, hr_img: Image.Image, size: int, rng: random.Random, candidates: int) -> tuple[Image.Image, Image.Image]:
    if candidates <= 1 or lr_img.width <= size or lr_img.height <= size:
        return random_crop_pair(lr_img, hr_img, size, rng)
    best: tuple[float, tuple[int, int]] | None = None
    lr_np_full = pil_to_np(lr_img)
    hr_np_full = pil_to_np(hr_img)
    for _ in range(candidates):
        x0 = rng.randint(0, lr_img.width - size)
        y0 = rng.randint(0, lr_img.height - size)
        score = np_psnr(lr_np_full[y0:y0 + size, x0:x0 + size], hr_np_full[y0:y0 + size, x0:x0 + size])
        if best is None or score < best[0]:
            best = (score, (x0, y0))
    assert best is not None
    x0, y0 = best[1]
    box = (x0, y0, x0 + size, y0 + size)
    return lr_img.crop(box), hr_img.crop(box)


def augment_pair(lr_img: Image.Image, hr_img: Image.Image, rng: random.Random) -> tuple[Image.Image, Image.Image]:
    if rng.random() > 0.5:
        lr_img = lr_img.transpose(Image.FLIP_LEFT_RIGHT)
        hr_img = hr_img.transpose(Image.FLIP_LEFT_RIGHT)
    if rng.random() > 0.5:
        lr_img = lr_img.transpose(Image.FLIP_TOP_BOTTOM)
        hr_img = hr_img.transpose(Image.FLIP_TOP_BOTTOM)
    k = rng.randint(0, 3)
    if k:
        op = {1: Image.ROTATE_90, 2: Image.ROTATE_180, 3: Image.ROTATE_270}[k]
        lr_img = lr_img.transpose(op)
        hr_img = hr_img.transpose(op)
    return lr_img, hr_img


def augment_single(img: Image.Image, rng: random.Random) -> Image.Image:
    if rng.random() > 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
    if rng.random() > 0.5:
        img = img.transpose(Image.FLIP_TOP_BOTTOM)
    k = rng.randint(0, 3)
    if k:
        img = img.transpose({1: Image.ROTATE_90, 2: Image.ROTATE_180, 3: Image.ROTATE_270}[k])
    return img



def jpeg_roundtrip(img: Image.Image, quality: int) -> Image.Image:
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=quality)
    buf.seek(0)
    return Image.open(buf).convert("RGB")


def apply_rgb_gamma_shift(img: Image.Image, rng: random.Random, strength: float = 1.0) -> Image.Image:
    arr = pil_to_np(img)
    gains = np.array([rng.uniform(1 - 0.035 * strength, 1 + 0.035 * strength) for _ in range(3)], dtype=np.float32)
    bias = np.array([rng.uniform(-0.010 * strength, 0.010 * strength) for _ in range(3)], dtype=np.float32)
    gamma = rng.uniform(1 - 0.070 * strength, 1 + 0.070 * strength)
    arr = np.clip(arr * gains + bias, 0.0, 1.0) ** gamma
    return Image.fromarray(np.uint8(np.clip(arr, 0.0, 1.0) * 255.0 + 0.5), mode="RGB")


def add_noise(img: Image.Image, rng: random.Random, sigma_range: tuple[float, float]) -> Image.Image:
    arr = pil_to_np(img)
    sigma = rng.uniform(*sigma_range)
    noise_rng = np.random.default_rng(rng.randint(0, 2**32 - 1))
    arr = np.clip(arr + noise_rng.normal(0.0, sigma, size=arr.shape), 0.0, 1.0)
    return Image.fromarray(np.uint8(arr * 255.0 + 0.5), mode="RGB")


def weighted_choice(rng: random.Random, items: list[tuple[str, float]]) -> str:
    total = sum(weight for _, weight in items)
    draw = rng.random() * total
    acc = 0.0
    for item, weight in items:
        acc += weight
        if draw <= acc:
            return item
    return items[-1][0]


def resize_degrade(img: Image.Image, rng: random.Random, scales: list[int], down_modes: list[int], up_modes: list[int]) -> tuple[Image.Image, int]:
    scale = rng.choice(scales)
    small = (max(1, img.width // scale), max(1, img.height // scale))
    return img.resize(small, resample=rng.choice(down_modes)).resize(img.size, resample=rng.choice(up_modes)), scale


def generate_degraded_candidate(hr_img: Image.Image, rng: random.Random) -> tuple[Image.Image, dict[str, Any]]:
    family = weighted_choice(rng, [
        ("blur_resize_jpeg", 0.38),
        ("color_residual", 0.22),
        ("noise_jpeg", 0.20),
        ("strong_resize_softblur", 0.20),
    ])
    lr_img = hr_img.copy()
    params: dict[str, Any] = {"recipe_family": family}
    if family == "blur_resize_jpeg":
        if rng.random() < 0.92:
            radius = rng.uniform(0.30, 1.90)
            lr_img = lr_img.filter(ImageFilter.GaussianBlur(radius=radius))
            params["blur_radius"] = radius
        lr_img, scale = resize_degrade(lr_img, rng, [2, 3, 4], [BICUBIC, BILINEAR, LANCZOS], [BICUBIC, BILINEAR])
        if rng.random() < 0.85:
            quality = rng.randint(24, 88)
            lr_img = jpeg_roundtrip(lr_img, quality)
            params["jpeg_quality"] = quality
    elif family == "color_residual":
        lr_img, scale = resize_degrade(lr_img, rng, [2, 3, 4], [BICUBIC, BILINEAR], [BICUBIC, BILINEAR])
        lr_img = apply_rgb_gamma_shift(lr_img, rng, strength=1.2)
        params["rgb_gamma_shift"] = True
        if rng.random() < 0.45:
            quality = rng.randint(42, 92)
            lr_img = jpeg_roundtrip(lr_img, quality)
            params["jpeg_quality"] = quality
    elif family == "noise_jpeg":
        lr_img, scale = resize_degrade(lr_img, rng, [2, 3, 4], [BICUBIC, BILINEAR, LANCZOS], [BICUBIC, BILINEAR])
        quality = rng.randint(20, 78)
        lr_img = jpeg_roundtrip(lr_img, quality)
        lr_img = add_noise(lr_img, rng, (0.003, 0.022))
        params["jpeg_quality"] = quality
        params["noise"] = True
    else:
        radius = rng.uniform(0.55, 2.45)
        lr_img = lr_img.filter(ImageFilter.GaussianBlur(radius=radius))
        lr_img, scale = resize_degrade(lr_img, rng, [3, 4, 5], [BICUBIC, BILINEAR], [BICUBIC, BILINEAR])
        if rng.random() < 0.55:
            quality = rng.randint(35, 90)
            lr_img = jpeg_roundtrip(lr_img, quality)
            params["jpeg_quality"] = quality
        params["blur_radius"] = radius
    params["scale"] = scale
    return lr_img, params


def degradation_score(row: dict[str, Any], cfg: dict[str, Any], target: dict[str, float]) -> float:
    psnr_penalty = abs(row["baseline_psnr"] - target["center"])
    grad_center = 0.5 * (cfg["target_grad_ratio_low"] + cfg["target_grad_ratio_high"])
    lap_center = 0.5 * (cfg["target_lap_ratio_low"] + cfg["target_lap_ratio_high"])
    score = psnr_penalty + 4.0 * abs(row["grad_ratio"] - grad_center) + 3.0 * abs(row["lap_ratio"] - lap_center)
    if cfg.get("target_mean_abs_residual", 0.0) > 0:
        score += 30.0 * abs(row["mean_abs_residual"] - cfg["target_mean_abs_residual"])
    if "target_rgb_residual_mean" in cfg:
        score += 50.0 * float(np.mean(np.abs(np.array(row["signed_residual_mean_rgb"]) - np.array(cfg["target_rgb_residual_mean"]))))
    return float(score)


def stage_target_bounds(cfg: dict[str, Any], stage_name: str) -> dict[str, float]:
    if stage_name == "mixed_finetune":
        low, high = cfg["stage2_target_psnr_low"], cfg["stage2_target_psnr_high"]
    else:
        low, high = cfg["target_psnr_low"], cfg["target_psnr_high"]
    return {"low": float(low), "high": float(high), "center": float(min(max(cfg["target_psnr_center"], low), high))}


def row_matches_target(row: dict[str, Any], cfg: dict[str, Any], target: dict[str, float]) -> bool:
    residual_ok = True
    if cfg.get("target_mean_abs_residual", 0.0) > 0:
        residual_ok = 0.45 * cfg["target_mean_abs_residual"] <= row["mean_abs_residual"] <= 1.90 * cfg["target_mean_abs_residual"]
    return (
        target["low"] <= row["baseline_psnr"] <= target["high"]
        and cfg["target_grad_ratio_low"] <= row["grad_ratio"] <= cfg["target_grad_ratio_high"]
        and cfg["target_lap_ratio_low"] <= row["lap_ratio"] <= cfg["target_lap_ratio_high"]
        and residual_ok
    )


def calibrated_degrade_from_hr(hr_img: Image.Image, rng: random.Random, cfg: dict[str, Any], stage_name: str = "synthetic_warmup") -> tuple[Image.Image, dict[str, Any]]:
    best: tuple[float, Image.Image, dict[str, Any]] | None = None
    target = stage_target_bounds(cfg, stage_name)
    for attempt in range(cfg["max_degrade_attempts"]):
        candidate, params = generate_degraded_candidate(hr_img, rng)
        row = pair_metric_row(candidate, hr_img, name="synthetic", source="synthetic")
        params = {**params, **{k: row[k] for k in ["baseline_psnr", "mean_abs_residual", "signed_residual_mean_rgb", "grad_ratio", "lap_ratio"]}, "attempt": attempt + 1}
        score = degradation_score(row, cfg, target)
        if best is None or score < best[0]:
            best = (score, candidate, params)
        if row_matches_target(row, cfg, target):
            params["accepted"] = True
            return candidate, params
    assert best is not None
    best[2]["accepted"] = False
    return best[1], best[2]


natural_records = build_natural_records(DATA_ROOT, DATA_CFG, SEED)


Natural records: train=18000 (COCO=12000, ImageNet=6000) | val=800


In [8]:
# Dataset and loader construction
class PairedSRDataset(Dataset):
    def __init__(self, pairs: list[tuple[Path, Path, str]], train: bool, data_cfg: dict[str, Any], seed: int, source_name: str = "paired_train", hard_mine: bool = False):
        self.pairs = pairs
        self.train = train
        self.data_cfg = data_cfg
        self.seed = seed
        self.source_name = source_name
        self.hard_mine = hard_mine

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx: int):
        lr_path, hr_path, name = self.pairs[idx]
        lr_img, hr_img = pil_rgb(lr_path), pil_rgb(hr_path)
        rng = random.Random(self.seed + idx + (100_000 if self.hard_mine else 0)) if self.train else seeded_rng(f"{self.source_name}:{name}")
        if self.train:
            if self.hard_mine:
                lr_img, hr_img = hard_mine_crop_pair(lr_img, hr_img, self.data_cfg["train_patch_size"], rng, self.data_cfg["patch_mining_candidates"])
            else:
                lr_img, hr_img = random_crop_pair(lr_img, hr_img, self.data_cfg["train_patch_size"], rng)
            lr_img, hr_img = augment_pair(lr_img, hr_img, rng)
        else:
            lr_img = ImageOps.fit(lr_img, (self.data_cfg["eval_size"], self.data_cfg["eval_size"]), method=BICUBIC)
            hr_img = ImageOps.fit(hr_img, (self.data_cfg["eval_size"], self.data_cfg["eval_size"]), method=BICUBIC)
        return TO_TENSOR(lr_img), TO_TENSOR(hr_img), name, self.source_name


class CalibratedSyntheticSRDataset(Dataset):
    def __init__(self, records: list[dict[str, Any]], train: bool, data_cfg: dict[str, Any], seed: int, source_name: str, stage_name: str, cache_dir: Path | None = None):
        self.records = records
        self.train = train
        self.data_cfg = data_cfg
        self.seed = seed
        self.source_name = source_name
        self.stage_name = stage_name
        self.cache_dir = cache_dir if (cache_dir is not None and not train and data_cfg.get("synthetic_val_cache", True)) else None
        self.last_params: list[dict[str, Any]] = []
        if self.cache_dir is not None:
            self.cache_dir.mkdir(parents=True, exist_ok=True)
            (self.cache_dir / "cache_meta.json").write_text(json.dumps({
                "stage_name": stage_name,
                "source_name": source_name,
                "target_psnr_low": data_cfg["target_psnr_low"],
                "target_psnr_high": data_cfg["target_psnr_high"],
                "target_mean_abs_residual": data_cfg.get("target_mean_abs_residual"),
            }, indent=2))

    def __len__(self):
        return len(self.records)

    def cache_path(self, idx: int, stem: str) -> Path | None:
        if self.cache_dir is None:
            return None
        return self.cache_dir / f"{idx:06d}_{slugify_name(stem)}.pt"

    def __getitem__(self, idx: int):
        record = self.records[idx]
        cache_path = self.cache_path(idx, record["stem"])
        if cache_path is not None and cache_path.exists():
            lr_t, hr_t, params = torch.load(cache_path, map_location="cpu")
            return lr_t, hr_t, record["stem"], self.source_name
        hr_img = pil_rgb(record["path"])
        rng = random.Random(self.seed + idx) if self.train else seeded_rng(f"{self.source_name}:{record['stem']}:{self.stage_name}")
        if self.train:
            base_size = max(self.data_cfg["eval_size"], self.data_cfg["train_patch_size"] + self.data_cfg["random_scale_pad"])
            hr_img = ImageOps.fit(hr_img, (base_size, base_size), method=BICUBIC)
            if self.data_cfg.get("synthetic_patch_mining_candidates", 1) > 1:
                crops = [random_crop_single(hr_img, self.data_cfg["train_patch_size"], rng) for _ in range(self.data_cfg["synthetic_patch_mining_candidates"])]
                scored: list[tuple[float, Image.Image]] = []
                for crop in crops:
                    lr_probe, params_probe = calibrated_degrade_from_hr(crop, rng, self.data_cfg, self.stage_name)
                    scored.append((abs(float(params_probe["baseline_psnr"]) - self.data_cfg["target_psnr_center"]), crop))
                hr_img = min(scored, key=lambda item: item[0])[1]
            else:
                hr_img = random_crop_single(hr_img, self.data_cfg["train_patch_size"], rng)
            hr_img = augment_single(hr_img, rng)
        else:
            hr_img = ImageOps.fit(hr_img, (self.data_cfg["eval_size"], self.data_cfg["eval_size"]), method=BICUBIC)
        lr_img, params = calibrated_degrade_from_hr(hr_img, rng, self.data_cfg, self.stage_name)
        lr_t, hr_t = TO_TENSOR(lr_img), TO_TENSOR(hr_img)
        if self.train and len(self.last_params) < 2048:
            self.last_params.append(params)
        if cache_path is not None:
            torch.save((lr_t, hr_t, params), cache_path)
        return lr_t, hr_t, record["stem"], self.source_name


def loader_kwargs(num_workers: int, pin_memory: bool) -> dict[str, Any]:
    kwargs: dict[str, Any] = {"num_workers": num_workers, "pin_memory": pin_memory}
    if num_workers > 0:
        kwargs["persistent_workers"] = True
        kwargs["prefetch_factor"] = PREFETCH_FACTOR
    return kwargs


def make_fixed_subset_loader(dataset: Dataset, subset_size: int, batch_size: int, seed: int, num_workers: int, pin_memory: bool) -> DataLoader:
    subset_size = min(subset_size, len(dataset))
    rng = random.Random(seed)
    indices = list(range(len(dataset)))
    rng.shuffle(indices)
    subset = Subset(dataset, indices[:subset_size])
    return DataLoader(subset, batch_size=batch_size, shuffle=False, **loader_kwargs(num_workers, pin_memory))


def paired_image_weights(train_rows: list[dict[str, Any]], threshold: float, hard_weight: float) -> list[float]:
    rows_by_name = {row["name"]: row for row in train_rows}
    weights = []
    for _, _, name in train_pairs:
        psnr = rows_by_name.get(name, {}).get("baseline_psnr", threshold)
        weights.append(1.0 + hard_weight * max(0.0, threshold - float(psnr)) / max(1.0, threshold - DATA_CFG["target_psnr_low"]))
    return weights


def make_stage_datasets(stage_name: str, seed: int) -> dict[str, Dataset]:
    paired_train = PairedSRDataset(train_pairs, train=True, data_cfg=DATA_CFG, seed=seed, source_name="paired_train", hard_mine=stage_name != "synthetic_warmup")
    paired_train_eval = PairedSRDataset(train_pairs, train=False, data_cfg=DATA_CFG, seed=seed, source_name="paired_train")
    paired_val = PairedSRDataset(val_pairs, train=False, data_cfg=DATA_CFG, seed=seed, source_name="paired_val")
    synth_cache = OUTPUT_DIR / MODEL_ID / "synthetic_val_cache" / stage_name if SYNTHETIC_VAL_CACHE else None
    synthetic_train = CalibratedSyntheticSRDataset(natural_records["train"], True, DATA_CFG, seed, "calibrated_synthetic", stage_name)
    synthetic_val = CalibratedSyntheticSRDataset(natural_records["val"], False, DATA_CFG, seed, "calibrated_synthetic_val", stage_name, cache_dir=synth_cache) if natural_records["val"] else synthetic_train

    if stage_name == "synthetic_warmup":
        train_dataset: Dataset = synthetic_train
        sampler = None
    elif stage_name == "mixed_finetune":
        train_dataset = ConcatDataset([paired_train, synthetic_train])
        paired_weights = paired_image_weights(degradation_profile["paired_train_rows"], DATA_CFG["hard_image_threshold"], DATA_CFG["hard_patch_weight"])
        synthetic_weight = max(0.05, (1.0 - DATA_CFG["paired_fraction_mixed"]) / max(DATA_CFG["paired_fraction_mixed"], 1e-6))
        weights = paired_weights + [synthetic_weight for _ in range(len(synthetic_train))]
        sampler = WeightedRandomSampler(weights, num_samples=max(len(train_dataset), len(paired_train) * 2), replacement=True)
    elif stage_name == "paired_polish":
        train_dataset = paired_train
        weights = paired_image_weights(degradation_profile["paired_train_rows"], DATA_CFG["hard_image_threshold"], DATA_CFG["hard_patch_weight"])
        sampler = WeightedRandomSampler(weights, num_samples=len(paired_train), replacement=True)
    else:
        raise ValueError(f"Unknown stage: {stage_name}")

    return {
        "train": train_dataset,
        "train_sampler": sampler,
        "train_eval": paired_train_eval,
        "paired_val": paired_val,
        "synthetic_val": synthetic_val,
        "calibration": ConcatDataset([paired_train_eval, synthetic_val]) if len(natural_records["val"]) else paired_train_eval,
    }


def make_stage_loaders(stage_name: str, seed: int) -> dict[str, Any]:
    datasets = make_stage_datasets(stage_name, seed)
    pin_memory = device.type == "cuda"
    train_loader = DataLoader(
        datasets["train"],
        batch_size=BATCH_SIZE,
        shuffle=datasets["train_sampler"] is None,
        sampler=datasets["train_sampler"],
        drop_last=True,
        **loader_kwargs(NUM_WORKERS, pin_memory),
    )
    return {
        **datasets,
        "train_loader": train_loader,
        "train_eval_loader": make_fixed_subset_loader(datasets["train_eval"], DATA_CFG["train_eval_subset_size"], BATCH_SIZE, seed, NUM_WORKERS, pin_memory),
        "paired_val_loader": DataLoader(datasets["paired_val"], batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs(NUM_WORKERS, pin_memory)),
        "synthetic_val_loader": make_fixed_subset_loader(datasets["synthetic_val"], min(DATA_CFG["train_eval_subset_size"], len(datasets["synthetic_val"])), BATCH_SIZE, seed, NUM_WORKERS, pin_memory),
        "calibration_loader": DataLoader(datasets["calibration"], batch_size=1, shuffle=False, **loader_kwargs(0, False)),
    }


In [9]:
# Models and NPU/module checks
class ConvPReLUBlock(nn.Module):
    def __init__(self, channels: int, kernel_size: int = 3, groups: int = 1):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size, padding=kernel_size // 2, groups=groups, bias=True)
        self.act = nn.PReLU(num_parameters=channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.conv(x))


def init_tail_small(tail: nn.Conv2d, scale: float = 1e-3) -> None:
    nn.init.kaiming_normal_(tail.weight, nonlinearity="linear")
    tail.weight.data.mul_(scale)
    if tail.bias is not None:
        nn.init.zeros_(tail.bias)


class ResidualConvSR(nn.Module):
    def __init__(self, channels: int = 96, kernel_pattern: tuple[int, ...] = (3,) * 18):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, channels, 3, padding=1, bias=True), nn.PReLU(num_parameters=channels))
        self.body = nn.Sequential(*[ConvPReLUBlock(channels, kernel_size=k) for k in kernel_pattern])
        self.tail = nn.Conv2d(channels, 3, 3, padding=1, bias=True)
        init_tail_small(self.tail)

    def predict_delta(self, x: torch.Tensor) -> torch.Tensor:
        return self.tail(self.body(self.stem(x)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.predict_delta(x)


class ExpandedDepthwiseBlock(nn.Module):
    def __init__(self, channels: int, expansion: int, kernel_size: int):
        super().__init__()
        hidden = channels * expansion
        self.net = nn.Sequential(
            nn.Conv2d(channels, hidden, 1, bias=True),
            nn.PReLU(num_parameters=hidden),
            nn.Conv2d(hidden, hidden, kernel_size, padding=kernel_size // 2, groups=hidden, bias=True),
            nn.PReLU(num_parameters=hidden),
            nn.Conv2d(hidden, channels, 1, bias=True),
            nn.PReLU(num_parameters=channels),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class ExpandedDepthwiseResidualSR(nn.Module):
    def __init__(self, channels: int = 128, expansion: int = 2, num_blocks: int = 12, kernel_size: int = 9):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, channels, 3, padding=1, bias=True), nn.PReLU(num_parameters=channels))
        self.body = nn.Sequential(*[ExpandedDepthwiseBlock(channels, expansion, kernel_size) for _ in range(num_blocks)])
        self.tail = nn.Conv2d(channels, 3, 3, padding=1, bias=True)
        init_tail_small(self.tail)

    def predict_delta(self, x: torch.Tensor) -> torch.Tensor:
        return self.tail(self.body(self.stem(x)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.predict_delta(x)


MODEL_REGISTRY = {
    "nonorm_residual": {
        "display_name": "ResidualConvSR_3x3x18",
        "build_model": lambda **cfg: ResidualConvSR(**cfg),
        "model_cfg": {"channels": 96, "kernel_pattern": tuple([3] * 18)},
    },
    "mixed_kernel_residual": {
        "display_name": "ResidualConvSR_mixed_3_3_5",
        "build_model": lambda **cfg: ResidualConvSR(**cfg),
        "model_cfg": {"channels": 96, "kernel_pattern": tuple([3, 3, 5] * 6)},
    },
    "expanded_dw_large_residual": {
        "display_name": "ExpandedDepthwiseResidualSR_9x9x12",
        "build_model": lambda **cfg: ExpandedDepthwiseResidualSR(**cfg),
        "model_cfg": {"channels": 128, "expansion": 2, "num_blocks": 12, "kernel_size": 9},
    },
    "expanded_dw_large_deep": {
        "display_name": "ExpandedDepthwiseResidualSR_7x7x18",
        "build_model": lambda **cfg: ExpandedDepthwiseResidualSR(**cfg),
        "model_cfg": {"channels": 128, "expansion": 2, "num_blocks": 18, "kernel_size": 7},
    },
}


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def assert_module_compatible(model: nn.Module) -> None:
    for name, module in model.named_modules():
        if isinstance(module, FORBIDDEN_MODULE_TYPES):
            raise TypeError(f"Forbidden module for Mobilint path at {name}: {module.__class__.__name__}")
        if isinstance(module, (nn.BatchNorm2d, nn.Dropout, nn.Dropout2d, nn.AdaptiveAvgPool2d, nn.Sigmoid, nn.Hardsigmoid)):
            raise TypeError(f"Phase 7B model should avoid normalization/attention/dropout at {name}: {module.__class__.__name__}")


def summarize_model_modules(model: nn.Module) -> dict[str, int]:
    counts: dict[str, int] = defaultdict(int)
    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            key = "DepthwiseConv2d" if module.groups == module.in_channels and module.in_channels == module.out_channels else "Conv2d"
            counts[key] += 1
        elif isinstance(module, nn.PReLU):
            counts["PReLU"] += 1
    return dict(counts)


def get_model_spec(model_id: str) -> dict[str, Any]:
    if model_id not in MODEL_REGISTRY:
        raise KeyError(f"Unknown MODEL_ID={model_id}; choose {sorted(MODEL_REGISTRY)}")
    spec = copy.deepcopy(MODEL_REGISTRY[model_id])
    probe = spec["build_model"](**spec["model_cfg"])
    assert_module_compatible(probe)
    spec["params"] = count_parameters(probe)
    spec["module_ops"] = summarize_model_modules(probe)
    return spec


for model_id in MODEL_REGISTRY:
    model_spec = get_model_spec(model_id)
    print(f"{model_id}: {model_spec['display_name']} params={model_spec['params']:,} modules={model_spec['module_ops']}")

spec = get_model_spec(MODEL_ID)
print(f"Selected model: {MODEL_ID} / {spec['display_name']}")


nonorm_residual: ResidualConvSR_3x3x18 params=1,501,827 modules={'Conv2d': 20, 'PReLU': 19}
mixed_kernel_residual: ResidualConvSR_mixed_3_3_5 params=2,386,563 modules={'Conv2d': 20, 'PReLU': 19}
expanded_dw_large_residual: ExpandedDepthwiseResidualSR_9x9x12 params=1,057,795 modules={'Conv2d': 26, 'PReLU': 37, 'DepthwiseConv2d': 12}
expanded_dw_large_deep: ExpandedDepthwiseResidualSR_7x7x18 params=1,435,651 modules={'Conv2d': 38, 'PReLU': 55, 'DepthwiseConv2d': 18}
Selected model: mixed_kernel_residual / ResidualConvSR_mixed_3_3_5


In [10]:
# Training, metrics, diagnostics
class EMA:
    def __init__(self, model: nn.Module, decay: float):
        self.decay = decay
        self.shadow = {name: param.detach().clone() for name, param in model.named_parameters() if param.requires_grad}
        self.backup: dict[str, torch.Tensor] = {}

    def update(self, model: nn.Module) -> None:
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name].mul_(self.decay).add_(param.detach(), alpha=1.0 - self.decay)

    def apply_shadow(self, model: nn.Module) -> None:
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.shadow:
                self.backup[name] = param.detach().clone()
                param.data.copy_(self.shadow[name])

    def restore(self, model: nn.Module) -> None:
        for name, param in model.named_parameters():
            if name in self.backup:
                param.data.copy_(self.backup[name])
        self.backup = {}


def optimizer_with_fallback(model: nn.Module, train_cfg: dict[str, Any]) -> torch.optim.Optimizer:
    kwargs = {"lr": train_cfg["lr"], "weight_decay": train_cfg["weight_decay"]}
    if torch.cuda.is_available():
        try:
            return AdamW(model.parameters(), fused=True, **kwargs)
        except (TypeError, RuntimeError):
            pass
    return AdamW(model.parameters(), **kwargs)


def lr_for_epoch(epoch: int, total: int, base_lr: float, warmup: int, min_ratio: float) -> float:
    if warmup > 0 and epoch < warmup:
        return base_lr * float(epoch + 1) / float(warmup)
    progress = (epoch - warmup) / max(1, total - warmup - 1)
    progress = min(max(progress, 0.0), 1.0)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return base_lr * min_ratio + (base_lr - base_lr * min_ratio) * cosine


def move_batch(lr_img: torch.Tensor, hr_img: torch.Tensor, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    lr_img = lr_img.to(device, non_blocking=True)
    hr_img = hr_img.to(device, non_blocking=True)
    if channels_last and device.type == "cuda":
        lr_img = lr_img.contiguous(memory_format=torch.channels_last)
        hr_img = hr_img.contiguous(memory_format=torch.channels_last)
    return lr_img, hr_img


def residual_target_l1_loss(sr_pred: torch.Tensor, lr_img: torch.Tensor, hr_img: torch.Tensor) -> torch.Tensor:
    return F.l1_loss(sr_pred - lr_img, hr_img - lr_img)


def compute_psnr(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    return tensor_psnr(pred.clamp(0.0, 1.0), target.clamp(0.0, 1.0))


def synthetic_stats_from_dataset(dataset: Dataset) -> dict[str, Any]:
    params = getattr(dataset, "last_params", [])
    if not params:
        return {"count": 0}
    keys = ["baseline_psnr", "mean_abs_residual", "grad_ratio", "lap_ratio"]
    out: dict[str, Any] = {"count": len(params), "accepted_fraction": sum(bool(row.get("accepted")) for row in params) / len(params)}
    recipe_counts: dict[str, int] = defaultdict(int)
    for row in params:
        recipe_counts[str(row.get("recipe_family", "unknown"))] += 1
    out["recipe_counts"] = dict(sorted(recipe_counts.items()))
    for key in keys:
        vals = [float(row[key]) for row in params if key in row]
        if vals:
            out[f"synthetic_{key}_mean"] = float(np.mean(vals))
            out[f"synthetic_{key}_p50"] = float(np.quantile(vals, 0.5))
    return out


def maybe_fail_on_easy_synthetic(stats: dict[str, Any], stage_name: str) -> None:
    mean_psnr = stats.get("synthetic_baseline_psnr_mean")
    if mean_psnr is None:
        return
    too_easy = stage_name in {"synthetic_warmup", "mixed_finetune"} and mean_psnr >= SYNTHETIC_TOO_EASY_PSNR
    stats["synthetic_too_easy"] = bool(too_easy)
    if not too_easy:
        return
    message = f"Calibrated synthetic data drifted too easy: mean baseline PSNR={mean_psnr:.3f} dB"
    stats["synthetic_warning"] = message
    if FAIL_ON_EASY_SYNTHETIC:
        raise RuntimeError(message)
    print(f"WARNING: {message}; continuing because LAB2_FAIL_ON_EASY_SYNTHETIC=0")


def maybe_compile_forward_model(model: nn.Module) -> nn.Module:
    if not TORCH_COMPILE or device.type != "cuda" or not hasattr(torch, "compile"):
        return model
    try:
        compiled = torch.compile(model)
        print("torch.compile enabled for training forward pass")
        return compiled
    except Exception as exc:
        print(f"torch.compile skipped: {exc}")
        return model


def train_one_epoch(forward_model: nn.Module, ema_model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer, train_cfg: dict[str, Any], ema: EMA | None, scaler=None) -> dict[str, float]:
    forward_model.train()
    total_loss, total_psnr, n = 0.0, 0.0, 0
    for step, (lr_img, hr_img, _, _) in enumerate(loader):
        if MAX_STEPS_PER_EPOCH > 0 and step >= MAX_STEPS_PER_EPOCH:
            break
        lr_img, hr_img = move_batch(lr_img, hr_img, device)
        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:
            with autocast_context(device, amp_policy):
                pred = forward_model(lr_img)
                loss = residual_target_l1_loss(pred, lr_img, hr_img)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(ema_model.parameters(), train_cfg["grad_clip_norm"])
            scaler.step(optimizer)
            scaler.update()
        else:
            with autocast_context(device, amp_policy):
                pred = forward_model(lr_img)
                loss = residual_target_l1_loss(pred, lr_img, hr_img)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(ema_model.parameters(), train_cfg["grad_clip_norm"])
            optimizer.step()
        if ema is not None:
            ema.update(ema_model)
        bs = lr_img.size(0)
        total_loss += float(loss.item()) * bs
        with torch.no_grad():
            total_psnr += compute_psnr(pred.detach(), hr_img).sum().item()
        n += bs
    return {"train_loss": total_loss / max(1, n), "train_psnr_online": total_psnr / max(1, n)}


@torch.no_grad()
def evaluate_loader(forward_model: nn.Module, loader: DataLoader, train_cfg: dict[str, Any], split_name: str) -> dict[str, float]:
    forward_model.eval()
    total_loss, total_psnr, total_input_psnr, total_delta, total_res_ratio, n = 0.0, 0.0, 0.0, 0.0, 0.0, 0
    hard_buckets = {"hard": [], "mid": [], "easy": []}
    worst: list[dict[str, Any]] = []
    for step, (lr_img, hr_img, names, sources) in enumerate(loader):
        if MAX_EVAL_BATCHES > 0 and step >= MAX_EVAL_BATCHES:
            break
        lr_img, hr_img = move_batch(lr_img, hr_img, device)
        with autocast_context(device, amp_policy):
            pred = forward_model(lr_img)
            loss = residual_target_l1_loss(pred, lr_img, hr_img)
        psnr = compute_psnr(pred, hr_img)
        input_psnr = compute_psnr(lr_img, hr_img)
        delta = psnr - input_psnr
        residual_pred = (pred - lr_img).abs().mean(dim=(1, 2, 3))
        residual_target = (hr_img - lr_img).abs().mean(dim=(1, 2, 3)).clamp_min(1e-8)
        residual_ratio = residual_pred / residual_target
        bs = lr_img.size(0)
        total_loss += float(loss.item()) * bs
        total_psnr += psnr.sum().item()
        total_input_psnr += input_psnr.sum().item()
        total_delta += delta.sum().item()
        total_res_ratio += residual_ratio.sum().item()
        n += bs
        for i in range(bs):
            inp = float(input_psnr[i].item())
            pred_p = float(psnr[i].item())
            item = {"name": str(names[i]), "source": str(sources[i]), "input_psnr": inp, "pred_psnr": pred_p, "delta_psnr": pred_p - inp, "residual_ratio": float(residual_ratio[i].item())}
            if split_name == "paired_val":
                bucket = "hard" if inp < 21.0 else ("mid" if inp < 24.5 else "easy")
                hard_buckets[bucket].append(pred_p)
                worst.append(item)
    metrics = {
        f"{split_name}_loss": total_loss / max(1, n),
        f"{split_name}_psnr": total_psnr / max(1, n),
        f"{split_name}_input_psnr": total_input_psnr / max(1, n),
        f"{split_name}_delta_psnr": total_delta / max(1, n),
        f"{split_name}_residual_ratio": total_res_ratio / max(1, n),
    }
    if split_name == "paired_val":
        for bucket, vals in hard_buckets.items():
            if vals:
                metrics[f"paired_val_{bucket}_psnr"] = float(np.mean(vals))
        metrics["paired_val_worst10"] = sorted(worst, key=lambda row: row["pred_psnr"])[:10]  # type: ignore[assignment]
    return metrics


def verify_residual_l1_batch(model: nn.Module, loader: DataLoader) -> None:
    lr_img, hr_img, _, _ = next(iter(loader))
    lr_img, hr_img = move_batch(lr_img, hr_img, device)
    pred = model(lr_img)
    loss = residual_target_l1_loss(pred, lr_img, hr_img)
    manual = F.l1_loss(pred - lr_img, hr_img - lr_img)
    torch.testing.assert_close(loss, manual)


def load_history(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]


def save_checkpoint(path: Path, model: nn.Module, optimizer: torch.optim.Optimizer, epoch: int, metrics: dict[str, Any], best_metric: float, best_epoch: int, model_cfg: dict[str, Any], train_cfg: dict[str, Any], ema: EMA | None, scaler=None) -> None:
    state = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "metrics": metrics,
        "best_metric": best_metric,
        "best_epoch": best_epoch,
        "model_cfg": model_cfg,
        "train_cfg": train_cfg,
        "data_cfg": DATA_CFG,
        "amp_policy": amp_policy,
    }
    if ema is not None:
        state["ema_shadow"] = ema.shadow
    if scaler is not None:
        state["scaler_state_dict"] = scaler.state_dict()
    torch.save(state, path)


def load_weights(model: nn.Module, checkpoint_path: Path, map_location: str | torch.device = "cpu", use_ema: bool = True) -> dict[str, Any]:
    ckpt = torch.load(checkpoint_path, map_location=map_location)
    model.load_state_dict(ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt)
    if use_ema and isinstance(ckpt, dict) and "ema_shadow" in ckpt:
        for name, param in model.named_parameters():
            if name in ckpt["ema_shadow"]:
                param.data.copy_(ckpt["ema_shadow"][name].to(param.device))
    return ckpt


def stage_dir(stage_name: str) -> Path:
    return OUTPUT_DIR / MODEL_ID / stage_name


def fit_stage(model: nn.Module, stage_name: str, init_checkpoint: Path | None = None) -> Path:
    train_cfg = TRAIN_CFG_BY_STAGE[stage_name]
    loaders = make_stage_loaders(stage_name, SEED + {"synthetic_warmup": 0, "mixed_finetune": 101, "paired_polish": 202}[stage_name])
    output_dir = stage_dir(stage_name)
    output_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = output_dir / "metrics.jsonl"
    last_ckpt = output_dir / "last.pt"
    best_ckpt = output_dir / "best.pt"
    selection_metric = train_cfg["selection_metric"]

    model = model.to(device)
    if channels_last and device.type == "cuda":
        model = model.to(memory_format=torch.channels_last)
    forward_model = maybe_compile_forward_model(model)
    optimizer = optimizer_with_fallback(model, train_cfg)
    ema = EMA(model, decay=train_cfg["ema_decay"])
    scaler = make_grad_scaler(amp_policy)
    start_epoch, best_metric, best_epoch = 0, float("-inf"), -1
    history: list[dict[str, Any]] = []

    if RESUME_TRAINING and last_ckpt.exists():
        ckpt = torch.load(last_ckpt, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        if "ema_shadow" in ckpt:
            ema.shadow = ckpt["ema_shadow"]
        if scaler is not None and "scaler_state_dict" in ckpt:
            scaler.load_state_dict(ckpt["scaler_state_dict"])
        start_epoch = int(ckpt["epoch"])
        best_metric = float(ckpt.get("best_metric", float("-inf")))
        best_epoch = int(ckpt.get("best_epoch", -1))
        history = load_history(metrics_path)
        print(f"Resuming {stage_name} from epoch {start_epoch}/{train_cfg['epochs']}")
    else:
        if init_checkpoint is not None and init_checkpoint.exists():
            load_weights(model, init_checkpoint, map_location=device, use_ema=True)
            ema = EMA(model, decay=train_cfg["ema_decay"])
            print(f"Initialized {stage_name} from {init_checkpoint}")
        metrics_path.write_text("")
        print(f"Fresh start: {stage_name}")

    print(f"\n{'epoch':>5} | {'lr':>8} | {'train':>8} {'train_eval':>10} | {'paired':>8} {'delta':>8} {'hard':>8} {'synthetic':>9} | {'best':>8} | {'time':>6}")
    print("-" * 118)
    epochs_without_improve = 0
    for epoch in range(start_epoch, train_cfg["epochs"]):
        epoch_num = epoch + 1
        epoch_lr = lr_for_epoch(epoch, train_cfg["epochs"], train_cfg["lr"], train_cfg["warmup_epochs"], train_cfg["min_lr_ratio"])
        for group in optimizer.param_groups:
            group["lr"] = epoch_lr
        start_time = time.time()
        train_metrics = train_one_epoch(forward_model, model, loaders["train_loader"], optimizer, train_cfg, ema, scaler=scaler)
        ema.apply_shadow(model)
        train_eval_metrics = evaluate_loader(forward_model, loaders["train_eval_loader"], train_cfg, "train_eval") if epoch_num == train_cfg["epochs"] or epoch_num % train_cfg["train_eval_interval"] == 0 else {}
        paired_metrics = evaluate_loader(forward_model, loaders["paired_val_loader"], train_cfg, "paired_val")
        run_synthetic_eval = stage_name != "paired_polish" and (epoch_num == train_cfg["epochs"] or epoch_num % SYNTHETIC_EVAL_INTERVAL == 0)
        synthetic_metrics = evaluate_loader(forward_model, loaders["synthetic_val_loader"], train_cfg, "synthetic_val") if run_synthetic_eval else {}
        ema.restore(model)
        synthetic_dataset = loaders["train"] if isinstance(loaders["train"], CalibratedSyntheticSRDataset) else (loaders["train"].datasets[1] if isinstance(loaders["train"], ConcatDataset) and len(loaders["train"].datasets) > 1 else loaders["train"])
        synthetic_stats = synthetic_stats_from_dataset(synthetic_dataset)
        maybe_fail_on_easy_synthetic(synthetic_stats, stage_name)
        row: dict[str, Any] = {
            "epoch": epoch_num,
            "lr": epoch_lr,
            "stage": stage_name,
            "model_id": MODEL_ID,
            "loss_target": "delta_hr_minus_lr_l1",
            **train_metrics,
            **train_eval_metrics,
            **paired_metrics,
            **synthetic_metrics,
            **synthetic_stats,
            "seconds": round(time.time() - start_time, 1),
            "selection_metric": selection_metric,
        }
        row["selection_metric_value"] = row[selection_metric]
        history.append(row)
        with metrics_path.open("a") as f:
            f.write(json.dumps(row) + "\n")
        metric_value = float(row[selection_metric])
        min_improvement = max(0.0, MIN_METRIC_IMPROVEMENT)
        improved = metric_value > (best_metric + min_improvement)
        if improved:
            best_metric, best_epoch, epochs_without_improve = metric_value, epoch_num, 0
        else:
            epochs_without_improve += 1
        save_checkpoint(last_ckpt, model, optimizer, epoch_num, row, best_metric, best_epoch, spec["model_cfg"], train_cfg, ema, scaler=scaler)
        if improved:
            save_checkpoint(best_ckpt, model, optimizer, epoch_num, row, best_metric, best_epoch, spec["model_cfg"], train_cfg, ema, scaler=scaler)
        if (SAVE_EPOCH_CHECKPOINTS or stage_name == "paired_polish") and epoch_num % train_cfg["checkpoint_interval"] == 0:
            save_checkpoint(output_dir / f"epoch_{epoch_num:03d}.pt", model, optimizer, epoch_num, row, best_metric, best_epoch, spec["model_cfg"], train_cfg, ema, scaler=scaler)
        print(f"{epoch_num:5d} | {epoch_lr:8.2e} | {row['train_psnr_online']:8.3f} {row.get('train_eval_psnr', float('nan')):10.3f} | {row['paired_val_psnr']:8.3f} {row['paired_val_delta_psnr']:8.3f} {row.get('paired_val_hard_psnr', float('nan')):8.3f} {row.get('synthetic_val_psnr', float('nan')):9.3f} | {best_metric:8.3f} | {row['seconds']:6.1f}{'*' if improved else ' '}")
        if epochs_without_improve >= train_cfg["early_stop_patience"]:
            print(f"Early stopping {stage_name} after {train_cfg['early_stop_patience']} epochs without >= {min_improvement:.4f} {selection_metric} improvement.")
            break
    summary = {"stage": stage_name, "best_epoch": best_epoch, "best_metric": best_metric, "best_ckpt": str(best_ckpt), "last_row": history[-1] if history else None}
    (output_dir / "summary.json").write_text(json.dumps(summary, indent=2))
    return best_ckpt


In [11]:
# Full training entrypoint
best_checkpoint_by_stage: dict[str, Path] = {}
if not RUN_FULL_TRAINING:
    raise RuntimeError("LAB2_RUN_FULL_TRAINING=0, but this notebook is configured to go straight to full training.")

model = spec["build_model"](**spec["model_cfg"])
init_ckpt = None
for stage in ["synthetic_warmup", "mixed_finetune", "paired_polish"]:
    best_ckpt = fit_stage(model, stage, init_checkpoint=init_ckpt)
    best_checkpoint_by_stage[stage] = best_ckpt
    init_ckpt = best_ckpt
FINAL_BEST_CKPT = best_checkpoint_by_stage["paired_polish"]


Fresh start: synthetic_warmup

epoch |       lr |    train train_eval |   paired    delta     hard synthetic |     best |   time
----------------------------------------------------------------------------------------------------------------------
    1 | 1.50e-04 |   23.499        nan |   21.336   -0.000   19.111       nan |   21.336 |  909.1*
    2 | 3.00e-04 |   23.105     26.315 |   21.336   -0.000   19.111       nan |   21.336 |  279.5 
    3 | 3.00e-04 |   23.504        nan |   21.335   -0.002   19.110       nan |   21.336 |  274.0 
    4 | 2.86e-04 |   23.843     26.292 |   21.335   -0.002   19.111    23.499 |   21.336 |  292.8 
    5 | 2.48e-04 |   24.298        nan |   21.366    0.030   19.135       nan |   21.366 |  283.6*
    6 | 1.93e-04 |   24.471     26.321 |   21.395    0.058   19.166       nan |   21.395 |  273.0*
    7 | 1.31e-04 |   24.573        nan |   21.359    0.023   19.157       nan |   21.395 |  280.1 
    8 | 7.60e-05 |   24.648     25.850 |   21.237   -0.099 

In [12]:
# Checkpoint averaging / SWA-style late checkpoint evaluation
def average_checkpoints(checkpoints: list[Path], output_path: Path) -> Path | None:
    checkpoints = [Path(p) for p in checkpoints if Path(p).exists()]
    if not checkpoints:
        return None
    avg_state: dict[str, torch.Tensor] = {}
    template: dict[str, Any] | None = None
    for idx, ckpt_path in enumerate(checkpoints):
        ckpt = torch.load(ckpt_path, map_location="cpu")
        template = ckpt if template is None else template
        source = ckpt.get("ema_shadow") or ckpt["model_state_dict"]
        for name, tensor in source.items():
            if not torch.is_tensor(tensor):
                continue
            tensor = tensor.detach().float()
            if idx == 0:
                avg_state[name] = tensor.clone()
            else:
                avg_state[name].add_(tensor)
    for tensor in avg_state.values():
        tensor.div_(len(checkpoints))
    assert template is not None
    averaged = dict(template)
    averaged["model_state_dict"] = template["model_state_dict"]
    averaged["ema_shadow"] = avg_state
    averaged["averaged_from"] = [str(p) for p in checkpoints]
    output_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(averaged, output_path)
    return output_path


def collect_late_checkpoints(stage_name: str, max_count: int = 4) -> list[Path]:
    root = stage_dir(stage_name)
    candidates = sorted(root.glob("epoch_*.pt")) + ([root / "best.pt"] if (root / "best.pt").exists() else [])
    unique = []
    seen = set()
    for path in candidates:
        if path not in seen:
            seen.add(path)
            unique.append(path)
    return unique[-max_count:]

if RUN_FULL_TRAINING:
    late = collect_late_checkpoints("paired_polish", max_count=4)
    avg_path = average_checkpoints(late, stage_dir("paired_polish") / "averaged_late.pt")
    if avg_path is not None:
        eval_model = spec["build_model"](**spec["model_cfg"]).to(device)
        load_weights(eval_model, avg_path, map_location=device, use_ema=True)
        loaders = make_stage_loaders("paired_polish", SEED + 404)
        avg_metrics = evaluate_loader(eval_model, loaders["paired_val_loader"], TRAIN_CFG_BY_STAGE["paired_polish"], "paired_val")
        (stage_dir("paired_polish") / "averaged_late_metrics.json").write_text(json.dumps(avg_metrics, indent=2))
        print(f"Averaged late checkpoint metrics: paired_val_psnr={avg_metrics['paired_val_psnr']:.3f}")
else:
    print("Checkpoint averaging skipped because full training did not run.")


In [13]:
# ONNX export, op audit, and calibration export
def audit_onnx_ops(onnx_path: Path, output_path: Path | None = None) -> dict[str, Any]:
    if onnx is None:
        print("onnx is not installed; skipping op audit")
        return {"status": "skipped", "reason": "onnx not installed"}
    model = onnx.load(str(onnx_path))
    counts: dict[str, int] = defaultdict(int)
    for node in model.graph.node:
        counts[node.op_type] += 1
    supported = {op: n for op, n in sorted(counts.items()) if op in MOBILINT_NPU_SUPPORTED_ONNX}
    cpu_fallback = {op: n for op, n in sorted(counts.items()) if op in MOBILINT_CPU_FALLBACK_ONNX}
    unknown = {op: n for op, n in sorted(counts.items()) if op not in MOBILINT_NPU_SUPPORTED_ONNX and op not in MOBILINT_CPU_FALLBACK_ONNX}
    audit = {"onnx_path": str(onnx_path), "counts": dict(sorted(counts.items())), "npu_supported": supported, "cpu_fallback_reported": cpu_fallback, "unknown": unknown}
    if output_path is not None:
        output_path.write_text(json.dumps(audit, indent=2))
    print("ONNX op audit:")
    print(json.dumps({"npu_supported": supported, "cpu_fallback_reported": cpu_fallback, "unknown": unknown}, indent=2))
    return audit


def export_to_onnx(checkpoint_path: Path, onnx_path: Path, verify: bool = True) -> Path:
    model = spec["build_model"](**spec["model_cfg"]).to(device)
    load_weights(model, checkpoint_path, map_location=device, use_ema=True)
    model.eval()
    dummy = torch.randn(1, 3, EVAL_SIZE, EVAL_SIZE, device=device)
    onnx_path.parent.mkdir(parents=True, exist_ok=True)
    export_kwargs = dict(export_params=True, opset_version=13, do_constant_folding=True, input_names=["input"], output_names=["output"])
    try:
        torch.onnx.export(model, dummy, str(onnx_path), dynamo=False, **export_kwargs)
    except TypeError:
        torch.onnx.export(model, dummy, str(onnx_path), **export_kwargs)
    if onnx is not None:
        onnx.checker.check_model(onnx.load(str(onnx_path)))
        print("ONNX checker passed.")
    if verify and ort is not None:
        sample_loader = make_stage_loaders("paired_polish", SEED + 505)["paired_val_loader"]
        sample_lr, _, _, _ = next(iter(sample_loader))
        sample_lr = sample_lr[:1].to(device)
        with torch.no_grad():
            torch_out = model(sample_lr).detach().cpu().numpy()
        sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
        ort_out = sess.run(["output"], {"input": sample_lr.cpu().numpy()})[0]
        diff = np.abs(torch_out - ort_out)
        print(f"ONNX Runtime parity: max_diff={diff.max():.8f}, mean_diff={diff.mean():.8f}")
    print(f"Exported ONNX: {onnx_path} ({onnx_path.stat().st_size / 1024:.0f} KB)")
    audit_onnx_ops(onnx_path, onnx_path.with_name("onnx_ops.json"))
    return onnx_path


@torch.no_grad()
def score_lr_tensor(lr_t: torch.Tensor, hr_t: torch.Tensor | None = None) -> dict[str, float]:
    gray = lr_t.float().mean(dim=0)
    grad_x = gray[:, 1:] - gray[:, :-1]
    grad_y = gray[1:, :] - gray[:-1, :]
    out = {
        "brightness": float(gray.mean().item()),
        "contrast": float(gray.std().item()),
        "texture": float(0.5 * (grad_x.abs().mean().item() + grad_y.abs().mean().item())),
    }
    if hr_t is not None:
        out["baseline_psnr"] = float(tensor_psnr(lr_t.unsqueeze(0), hr_t.unsqueeze(0))[0].item())
    return out


def select_calibration_records(loader: DataLoader, num_samples: int, seed: int) -> list[dict[str, Any]]:
    records = []
    for idx, (lr_t, hr_t, names, sources) in enumerate(loader):
        lr = lr_t[0]
        hr = hr_t[0]
        records.append({"loader_index": idx, "name": str(names[0]), "source": str(sources[0]), **score_lr_tensor(lr, hr)})
    if not records:
        return []
    for metric in ["brightness", "texture", "baseline_psnr"]:
        vals = np.array([row[metric] for row in records], dtype=np.float32)
        lo, hi = np.quantile(vals, [1 / 3, 2 / 3])
        for row in records:
            row[f"{metric}_bin"] = "low" if row[metric] <= lo else ("high" if row[metric] >= hi else "mid")
    rng = random.Random(seed)
    buckets: dict[tuple[str, str, str], list[dict[str, Any]]] = defaultdict(list)
    for row in records:
        buckets[(row["source"], row["brightness_bin"], row["texture_bin"])].append(row)
    for bucket in buckets.values():
        rng.shuffle(bucket)
    selected, seen = [], set()
    keys = sorted(buckets)
    while len(selected) < min(num_samples, len(records)):
        progress = False
        for key in keys:
            while buckets[key]:
                row = buckets[key].pop()
                if row["loader_index"] in seen:
                    continue
                selected.append(row)
                seen.add(row["loader_index"])
                progress = True
                break
            if len(selected) >= min(num_samples, len(records)):
                break
        if not progress:
            break
    return selected


def export_calibration_artifacts(output_dir: Path, cfg: dict[str, Any]) -> Path:
    loaders = make_stage_loaders("paired_polish", SEED + 606)
    loader = loaders["calibration_loader"]
    selected = select_calibration_records(loader, cfg["num_samples"], cfg["seed"])
    by_index = {row["loader_index"]: row for row in selected}
    cal_dir = output_dir / cfg["output_subdir"]
    cal_dir.mkdir(parents=True, exist_ok=True)
    batch = []
    manifest = []
    for idx, (lr_t, hr_t, names, sources) in enumerate(loader):
        if idx not in by_index:
            continue
        row = by_index[idx]
        lr = lr_t[0]
        batch.append(lr)
        image_path = cal_dir / f"{len(manifest):03d}_{slugify_name(row['source'])}_{slugify_name(row['name'])}.png"
        TO_PIL(lr).save(image_path)
        manifest.append({**row, "image_path": str(image_path)})
        if len(manifest) >= len(selected):
            break
    tensor_path = cal_dir / "calibration_inputs.pt"
    if batch:
        torch.save(torch.stack(batch), tensor_path)
    summary = summarize_numeric(manifest, ["brightness", "contrast", "texture", "baseline_psnr"]) if manifest else {"count": 0}
    (cal_dir / "manifest.json").write_text(json.dumps({"config": cfg, "summary": summary, "samples": manifest}, indent=2))
    print(f"Exported calibration artifacts: {cal_dir} ({len(manifest)} samples)")
    return cal_dir

if RUN_ONNX_EXPORT:
    if FINAL_BEST_CKPT is None:
        candidate = stage_dir("paired_polish") / "best.pt"
        if candidate.exists():
            FINAL_BEST_CKPT = candidate
    if FINAL_BEST_CKPT is not None and Path(FINAL_BEST_CKPT).exists():
        onnx_path = export_to_onnx(Path(FINAL_BEST_CKPT), stage_dir("paired_polish") / "best.onnx", verify=True)
        export_calibration_artifacts(onnx_path.parent, CALIBRATION_CFG)
    else:
        print("ONNX export skipped because no final checkpoint exists yet. Run full training first or place best.pt under the paired_polish stage dir.")
else:
    print("ONNX export skipped by configuration.")
